# Cluster Analysis of Selected Language Sample

### Common Setup

In [1]:
# Setting system path and project root
import os
import sys

PROJECT_ROOT_DIR = os.path.abspath('../../')
sys.path.append(PROJECT_ROOT_DIR) # bringing system path to project root

def get_fp(relative_path):
    return os.path.join(PROJECT_ROOT_DIR, relative_path)

### Reading Language Sample with Resource Category Data

In [2]:
sample_files = [
    "data_dir/fineweb2_metadata/l2v_nested_samples/5.csv",
    "data_dir/fineweb2_metadata/l2v_nested_samples/10.csv",
    "data_dir/fineweb2_metadata/l2v_nested_samples/22.csv",
    "data_dir/fineweb2_metadata/l2v_nested_samples/46.csv",
    "data_dir/fineweb2_metadata/l2v_nested_samples/96.csv",
    "data_dir/fineweb2_metadata/l2v_nested_samples/200.csv",
]

In [3]:
import warnings

# Ignore the specific warning emitted by lang2vec
warnings.filterwarnings(
    "ignore",
    message=r"pkg_resources is deprecated as an API.*",
    category=UserWarning,
    module=r"lang2vec\.lang2vec"
)

from lang2vec.lang2vec import *
def get_dist_matrix(lang_codes, distance_name):
        with zf(DISTANCES_FILE, 'r') as zp:
                data = sparse.load_npz(zp.open(map_distance_to_filename(distance_name)))

        present_idxs = np.array([DISTANCE_LANGUAGES.index(code) for code in lang_codes])
        dist_mat = [[data[lang2_idx, lang1_idx] if lang1_idx > lang2_idx else data[lang1_idx, lang2_idx]  for lang2_idx in present_idxs] for lang1_idx in present_idxs]
        return dist_mat

In [ ]:
# Fetch distance matrix
eval_distances = [
    "genetic",
    "geographic", 
    "syntactic",
    "inventory",
    "phonological",
    "featural"
]

available_langs = set(DISTANCE_LANGUAGES)  # all lang2vec-supported language codes

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import (silhouette_score, adjusted_rand_score,
                             normalized_mutual_info_score,
                             calinski_harabasz_score, davies_bouldin_score)
from sklearn.manifold import MDS
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
import scipy.spatial.distance as ssd


def normalize_matrix(mat):
    arr = np.asarray(mat, dtype=float)
    mn, mx = arr.min(), arr.max()
    return (arr - mn) / (mx - mn + 1e-12)


def build_coassociation_matrix(all_labels, n):
    """C[i,j] = fraction of base clusterings where i and j are in the same cluster."""
    C = np.zeros((n, n))
    for labels in all_labels:
        labels = np.asarray(labels)
        for cluster_id in np.unique(labels):
            idx = np.where(labels == cluster_id)[0]
            C[np.ix_(idx, idx)] += 1
    C /= len(all_labels)
    return C


def norm01(arr):
    arr = np.array(arr, dtype=float)
    return (arr - arr.min()) / (arr.max() - arr.min() + 1e-12)


def run_cluster_analysis(sample_lang_doc_distr_df):
    rng = np.random.default_rng(42)  # seeded RNG for reproducibility

    print(f'Shape of sample dataframe: {sample_lang_doc_distr_df.shape}')

    columns_of_interest = ["subset", "code", "name", "script", "family"]
    for col in columns_of_interest:
        if col in sample_lang_doc_distr_df.columns:
            uniques = sample_lang_doc_distr_df[col].dropna().unique()
            print(f"\nUnique values in '{col}' ({len(uniques)} total):")
        else:
            print(f"\nColumn '{col}' not found in the CSV.")

    lang_codes = sample_lang_doc_distr_df['code'].astype(str).unique()
    n = len(lang_codes)

    dist_matrix_map = {distance_name: get_dist_matrix(lang_codes, distance_name)
                       for distance_name in eval_distances}

    # Visualize distribution of each distance matrix
    for name, mat in dist_matrix_map.items():
        mat_arr   = np.asarray(mat, dtype=float)
        tril_vals = mat_arr[np.tril_indices_from(mat_arr, k=-1)]

        plt.figure(figsize=(6, 4))
        plt.hist(tril_vals, bins=30, range=(0, 1), density=True,
                 edgecolor='black', alpha=0.6, label='Histogram')
        if np.var(tril_vals) > 0:
            from scipy.stats import gaussian_kde
            jittered = tril_vals + rng.normal(scale=1e-12, size=tril_vals.shape)  # avoids KDE bandwidth=0 edge case
            kde = gaussian_kde(jittered)
            xs = np.linspace(0, 1, 200)
            plt.plot(xs, kde(xs), color='red', lw=2, label='Density')
        else:
            plt.text(0.5, 0.5, 'All values identical – no KDE',
                     transform=plt.gca().transAxes,
                     ha='center', va='center', color='gray')
        plt.title(f'Distance distribution – {name}')
        plt.xlabel('Distance')
        plt.ylabel('Density')
        plt.xlim(0, 1)
        plt.legend()
        plt.tight_layout()
        plt.show()

    distance_names = list(dist_matrix_map.keys())

    # Normalize each matrix to [0, 1] so no single distance type dominates
    norm_mats = {}
    for name in distance_names:
        mat = normalize_matrix(dist_matrix_map[name])
        mat = (mat + mat.T) / 2   # ensure exact symmetry
        np.fill_diagonal(mat, 0)
        norm_mats[name] = mat

    # Ensemble over multiple k values: yields fractional co-association instead of binary 0/1
    base_k_range = range(2, max(3, int(np.sqrt(n)) + 3))
    all_base_labels = []
    for k in base_k_range:
        for name in distance_names:
            agg = AgglomerativeClustering(n_clusters=k, metric='precomputed', linkage='average')
            all_base_labels.append(agg.fit_predict(norm_mats[name]))

    print(f"Built co-association from {len(all_base_labels)} base clusterings "
          f"(k={base_k_range.start}..{base_k_range.stop - 1} × {len(distance_names)} matrices)")

    C = build_coassociation_matrix(all_base_labels, n)
    consensus_dist = 1.0 - C  # high co-association = small distance
    np.fill_diagonal(consensus_dist, 0)

    # Build full hierarchy once; fcluster extracts any k without re-running
    condensed = ssd.squareform(consensus_dist)
    Z = linkage(condensed, method='average')

    # Dendrogram
    plt.figure(figsize=(max(14, n // 4), 6))
    if n <= 40:
        dendrogram(Z, labels=lang_codes, leaf_rotation=90, leaf_font_size=8)
    else:
        # Truncate to top 40 nodes for readability on large samples
        dendrogram(Z, truncate_mode='lastp', p=40, leaf_rotation=90,
                   leaf_font_size=8, show_contracted=True)
    plt.title('Language Dendrogram (Consensus Clustering)')
    plt.ylabel('Merge distance  (1 – co-association fraction)')
    plt.tight_layout()
    plt.show()

    # Natural k from largest linkage gap: biggest merge cost jump = most natural cut point
    max_k_eval = min(n - 1, 50)
    relevant_merges = Z[-max_k_eval:, 2]  # merge costs from max_k_eval+1 down to 1 cluster
    diffs = np.diff(relevant_merges)
    suggested_k = max_k_eval - int(np.argmax(diffs))
    print(f"Suggested k from largest linkage gap: {suggested_k}")

    # Multi-metric evaluation: embed into Euclidean space for CH and DB (they require coordinates)
    # n_init=1: no benefit from restarts when dissimilarity is precomputed
    n_components = min(n - 1, 20)
    mds = MDS(n_components=n_components, dissimilarity='precomputed',
              random_state=42, max_iter=500, n_init=1)
    X_embedded = mds.fit_transform(consensus_dist)

    range_k = list(range(2, max_k_eval + 1))
    sil_scores, ch_scores, db_scores = [], [], []

    for k in range_k:
        labels = fcluster(Z, t=k, criterion='maxclust') - 1
        if len(np.unique(labels)) < 2:
            sil_scores.append(0.0); ch_scores.append(0.0); db_scores.append(np.inf)
            continue
        sil_scores.append(silhouette_score(consensus_dist, labels, metric='precomputed'))
        ch_scores.append(calinski_harabasz_score(X_embedded, labels))
        db_scores.append(davies_bouldin_score(X_embedded, labels))

    # Combine: higher silhouette + higher CH + lower DB = better
    combined = (norm01(sil_scores) + norm01(ch_scores) + (1 - norm01(db_scores))) / 3

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    metrics = [
        (sil_scores, 'Silhouette  (higher = better)',         True),
        (ch_scores,  'Calinski-Harabász  (higher = better)',  True),
        (db_scores,  'Davies-Bouldin  (lower = better)',      False),
        (combined,   'Combined score  (higher = better)',     True),
    ]
    for ax, (scores, title, higher_better) in zip(axes.flat, metrics):
        ax.plot(range_k, scores, marker='o', markersize=4)
        best_idx = np.argmax(scores) if higher_better else np.argmin(scores)
        ax.axvline(range_k[best_idx], color='red', linestyle='--', alpha=0.7,
                   label=f'Best k = {range_k[best_idx]}')
        ax.axvline(suggested_k, color='green', linestyle=':', alpha=0.7,
                   label=f'Dendrogram gap k = {suggested_k}')
        ax.set_xlabel('Number of clusters (k)')
        ax.set_title(title)
        ax.grid(True, alpha=0.4)
        ax.legend(fontsize=8)
        ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
    plt.suptitle('Cluster quality metrics', fontsize=13)
    plt.tight_layout()
    plt.show()

    best_k_sil      = range_k[int(np.argmax(sil_scores))]
    best_k_ch       = range_k[int(np.argmax(ch_scores))]
    best_k_db       = range_k[int(np.argmin(db_scores))]
    best_k_combined = range_k[int(np.argmax(combined))]

    print(f"Best k — Silhouette:        {best_k_sil}  (usually biased toward 2, treat as lower bound)")
    print(f"Best k — Calinski-Harabász: {best_k_ch}")
    print(f"Best k — Davies-Bouldin:    {best_k_db}")
    print(f"Best k — Dendrogram gap:    {suggested_k}")
    print(f"Best k — Combined:          {best_k_combined} (recommended)")

    best_k = best_k_combined

    # External validation vs language family labels
    if 'family' in sample_lang_doc_distr_df.columns:
        final_labels = fcluster(Z, t=best_k, criterion='maxclust') - 1

        code_to_family = dict(zip(
            sample_lang_doc_distr_df['code'].astype(str),
            sample_lang_doc_distr_df['family'].astype(str)
        ))
        true_families_str = [code_to_family.get(c, 'unknown') for c in lang_codes]
        # np.unique avoids pd.factorize deprecation for non-Series inputs
        _, true_families = np.unique(np.asarray(true_families_str), return_inverse=True)

        # Suppress sklearn warning about integer labels being mistaken for regression targets
        with warnings.catch_warnings():
            warnings.filterwarnings(
                "ignore",
                message=r"The number of unique classes.*",
                category=UserWarning,
                module=r"sklearn\.metrics\.cluster\._supervised",
            )
            ari = adjusted_rand_score(true_families, final_labels)
            nmi = normalized_mutual_info_score(true_families, final_labels)

        print(f"\nExternal validation (k={best_k}) vs language family:")
        print(f"  ARI: {ari:.4f}  [range: -1..1, higher = better]")
        print(f"  NMI: {nmi:.4f}  [range: 0..1, higher = better]")


# Loop over the files and call the function
for rel_path in sample_files:
    csv_path = get_fp(rel_path)
    df = pd.read_csv(csv_path)
    print(f"\n>>>>>>>>>>>>>>>>>>>>>>>> Running analysis for: {rel_path} <<<<<<<<<<<<<<<<<<<<<<<<")
    run_cluster_analysis(df)


In [6]:
## Obsolete testing code
# DISTANCE_LANGUAGES
# data[0, 4]
# #dist_mat
# genetic_distance('rus','ajz')
# present_codes
# random_index = 44
# lang_sel = present_codes[random_index]
# print(lang_sel)
# lang_ind = present_idxs[random_index]
# print(lang_ind)
# print(dist_mat[random_index])
# ext_gen_dist = [genetic_distance(lang, present_codes[random_index]) for lang in present_codes]
# print(ext_gen_dist)
# print(ext_gen_dist == dist_mat[random_index])
# dist_mat = np.asarray(dist_mat, dtype=np.float64)
# print(dist_mat)